In [1]:
import anthropic

import os
import networkx as nx
from sentence_transformers import SentenceTransformer
import lancedb
import dspy

import grpc
import json
from senzing import SzEngine
from senzing_grpc import SzAbstractFactoryGrpc
from senzing import SzEngineFlags

print("All imports successful")

All imports successful


In [2]:
# Load the same embedding model used to create vectors
print("Loading embedding model...")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [3]:
db = lancedb.connect('/workspace/lancedb_data')
table = db.open_table('entities')

print(f"Connected to LanceDB")
print(f"Total entities available: {table.count_rows()}")

Connected to LanceDB
Total entities available: 196


In [4]:
G = nx.Graph()

# Get all entities
all_entities = table.to_pandas()

SENZING_HOST = os.getenv('SENZING_GRPC_HOST', 'senzing')
SENZING_PORT = os.getenv('SENZING_GRPC_PORT', '8261')

grpc_url = f"{SENZING_HOST}:{SENZING_PORT}"
grpc_channel = grpc.insecure_channel(grpc_url)
sz_abstract_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_abstract_factory.create_engine()

# Quick export to rebuild graph
entities = []
export_handle = sz_engine.export_json_entity_report(
    flags=SzEngineFlags.SZ_EXPORT_INCLUDE_ALL_ENTITIES | 
          SzEngineFlags.SZ_ENTITY_INCLUDE_RECORD_JSON_DATA |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ALL_RELATIONS
)

while True:
    try:
        entity_json = sz_engine.fetch_next(export_handle)
        if not entity_json:
            break
        entities.append(json.loads(entity_json))
    except StopIteration:
        break

sz_engine.close_export_report(export_handle)

# Build graph nodes
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    records = entity_data.get('RECORDS', [])
    
    if not records:
        continue
    
    # Get entity info from FEATURES (v4 format)
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    features = json_data.get('FEATURES', [])
    
    record_type = 'UNKNOWN'
    name = None
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    if not name:
        name = f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    
    G.add_node(
        entity_id,
        name=name,
        type=record_type,
        num_records=len(records),
        data_sources=data_sources
    )

# Build graph edges from Senzing's RELATED_ENTITIES
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    
    for rel in entity.get('RELATED_ENTITIES', []):
        related_id = rel.get('ENTITY_ID')
        match_key = rel.get('MATCH_KEY', 'related')
        
        if entity_id < related_id and G.has_node(related_id):
            G.add_edge(
                entity_id,
                related_id,
                relationship=match_key
            )

# Close Senzing
grpc_channel.close()

print(f"Knowledge graph rebuilt:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")

Knowledge graph rebuilt:
  Nodes: 196
  Edges: 492


In [5]:
entity_data = []

for entity in entities:
    entity_data_item = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data_item.get('ENTITY_ID')
    records = entity_data_item.get('RECORDS', [])
    
    if not records:
        continue
    
    # Get entity info from FEATURES (v4 format)
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    features = json_data.get('FEATURES', [])
    
    name = None
    record_type = 'UNKNOWN'
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    if not name:
        name = f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    
    # Get addresses from FEATURES (v4) and fallback to ADDRESSES
    addresses = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        for feat in rec_json.get('FEATURES', []):
            if feat.get('ADDR_FULL') and len(addresses) < 3:
                addresses.append(feat['ADDR_FULL'])
        for addr in rec_json.get('ADDRESSES', []):
            if addr.get('ADDR_FULL') and len(addresses) < 3:
                addresses.append(addr['ADDR_FULL'])
    
    # Get identifiers from FEATURES (v4) and fallback to IDENTIFIERS
    identifiers = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        for feat in rec_json.get('FEATURES', []):
            id_type = feat.get('NATIONAL_ID_TYPE') or feat.get('OTHER_ID_TYPE')
            id_num = feat.get('NATIONAL_ID_NUMBER') or feat.get('OTHER_ID_NUMBER')
            if id_type and id_num and len(identifiers) < 3:
                identifiers.append(f"{id_type}: {id_num}")
        for id_obj in rec_json.get('IDENTIFIERS', []):
            id_type = id_obj.get('NATIONAL_ID_TYPE') or id_obj.get('OTHER_ID_TYPE')
            id_num = id_obj.get('NATIONAL_ID_NUMBER') or id_obj.get('OTHER_ID_NUMBER')
            if id_type and id_num and len(identifiers) < 3:
                identifiers.append(f"{id_type}: {id_num}")
    
    # Get risks from top-level fields (these are outside FEATURES)
    risks = []
    for rec in records:
        rec_json = rec.get('JSON_DATA', {})
        risk_topics = rec_json.get('risk_topics', '')
        if risk_topics:
            risks.extend(risk_topics.split(','))
        for risk in rec_json.get('RISKS', []):
            topic = risk.get('TOPIC', '')
            if topic:
                risks.append(topic)
    
    # Get related entity info for richer text
    related = entity.get('RELATED_ENTITIES', [])
    
    # Create text description for embedding
    text_parts = [
        f"Name: {name}",
        f"Type: {record_type}",
        f"Data sources: {', '.join(data_sources)}",
        f"Records merged: {len(records)}"
    ]
    
    if addresses:
        text_parts.append(f"Address: {addresses[0]}")
    
    if identifiers:
        text_parts.append(f"Identifiers: {', '.join(identifiers[:2])}")
    
    if risks:
        text_parts.append(f"Risk topics: {', '.join(set(risks))}")
    
    if related:
        text_parts.append(f"Related to {len(related)} other entities")
    
    text = ". ".join(text_parts)
    
    # Create embedding
    embedding = embedding_model.encode(text).tolist()
    
    # Store data
    entity_data.append({
        'entity_id': entity_id,
        'name': name,
        'type': record_type,
        'text': text,
        'vector': embedding,
        'data_sources': ','.join(data_sources),
        'num_records': len(records),
        'addresses': '|'.join(addresses[:3]),
        'identifiers': '|'.join(identifiers[:3]),
        'risks': '|'.join(set(risks))
    })
    
    if len(entity_data) % 50 == 0:
        print(f"  Processed {len(entity_data)} entities...", end='\r')

print(f"\nCreated embeddings for {len(entity_data)} entities")

  Processed 150 entities...
Created embeddings for 196 entities


In [6]:
db = lancedb.connect('/workspace/lancedb_data')

# Drop existing table if it exists
try:
    db.drop_table('entities')
    print("Dropped existing table")
except:
    pass

# Create new table
print("Creating new table...")
table = db.create_table('entities', entity_data)

print(f"Stored {len(entity_data)} entities in LanceDB")
print("\nData preparation complete!")
print("You can now use the RAG notebook to query this data.")

Dropped existing table
Creating new table...
Stored 196 entities in LanceDB

Data preparation complete!
You can now use the RAG notebook to query this data.


In [7]:
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found in environment")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client configured with Claude Sonnet")

Anthropic client configured with Claude Sonnet


In [8]:
def ask_knowledge_graph(question, top_k=10):
    """
    Simple RAG: Search -> Expand -> Format -> Ask LLM
    """
    print(f"\nQuestion: {question}")
    print("="*70)
    
    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    
    print(f"Found {len(results)} relevant entities")
    
    # Step 2: Collect entity IDs and expand to neighbors
    entity_ids = set()
    for r in results:
        entity_ids.add(r['entity_id'])
        
        if r['entity_id'] in G:
            neighbors = list(G.neighbors(r['entity_id']))[:5]
            entity_ids.update(neighbors)
    
    print(f"Expanded to {len(entity_ids)} entities (including neighbors)")
    
    # Step 3: Build simple context
    context_parts = ["ENTITIES:\n"]
    
    for entity_id in list(entity_ids)[:30]:
        entity_info = table.search().where(f"entity_id = {entity_id}").limit(1).to_list()
        if not entity_info:
            continue
        
        info = entity_info[0]
        context_parts.append(f"- {info['name']} ({info['type']})")
        context_parts.append(f"  Sources: {info['data_sources']}, Records: {info['num_records']}")
        
        if info.get('risks'):
            context_parts.append(f"  Risks: {info['risks']}")
        
        if entity_id in G:
            rels = []
            for neighbor_id in list(G.neighbors(entity_id))[:3]:
                edge_data = G.get_edge_data(entity_id, neighbor_id)
                neighbor = G.nodes[neighbor_id]
                rel_type = edge_data.get('relationship', 'connected to') if edge_data else 'connected to'
                rels.append(f"{rel_type} {neighbor['name']}")
            
            if rels:
                context_parts.append(f"  Relationships: {'; '.join(rels)}")
        
        context_parts.append("")
    
    context = "\n".join(context_parts)
    
    # Step 4: Ask LLM
    print("Querying LLM...")
    
    message = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2048,
        system="Answer questions about a corporate ownership and sanctions knowledge graph.",
        messages=[
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    
    print("\n" + "="*70)
    print("ANSWER")
    print("="*70)
    print(message.content[0].text)
    print("="*70)

In [9]:
print("Knowledge Graph RAG - Interactive Session")
print("="*70)
print("Ask any question about the corporate ownership and sanctions data.")
print("The system will search LanceDB and query the knowledge graph.")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    
    if not question:
        continue
    
    try:
        ask_knowledge_graph(question)
        print()
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Knowledge Graph RAG - Interactive Session
Ask any question about the corporate ownership and sanctions data.
The system will search LanceDB and query the knowledge graph.
Type 'quit' to exit.



Your question:  do you see any signatures of fraud in this dataset?



Question: do you see any signatures of fraud in this dataset?
Found 10 relevant entities
Expanded to 46 entities (including neighbors)
Querying LLM...

ANSWER
Based on the knowledge graph provided, I can identify several patterns that warrant scrutiny, though they don't definitively prove fraud:

## 🚩 Red Flags for Potential Investigation

### 1. **Sanctions Evasion Risk - Said Kerimov Network**
- **Said Kerimov** is sanctioned (marked with "sanction" risk)
- **NUGGET CAPITAL PLC** has "sanction.linked" risk
- **TMF CORPORATE SERVICES LIMITED** acts as intermediary between Said Kerimov and NUGGET CAPITAL PLC
- **AIRCRAFT 32A-3454 LIMITED** also connects to both Said Kerimov and NUGGET CAPITAL PLC
- **POLYUS CAPITAL LIMITED** is related to Said Kerimov

**Concern**: This network structure could facilitate sanctions evasion through corporate layering and nominee services.

### 2. **Identity Confusion - Multiple "Daniel" Variations**
Multiple individuals with similar names appear across 

Your question:  q


Goodbye!
